##RAG architecture
A typical RAG application has two main components:

* **Indexing:** A pipeline for ingesting and indexing data from a source. This usually happens offline.
* **Retrieval and generation:** The actual RAG chain takes the user query at run time and retrieves the relevant data from the index, then passes that to the model.

The most common full sequence from raw data to answer looks like the following examples.

* **Indexing**
**Load:** First, you must load your data. This is done with DocumentLoaders.

**Split:** Text splitters break large Documents into smaller chunks. This is useful both for indexing data and for passing it into a model because large chunks are harder to search and won’t fit in a model’s finite context window.

**Store:** You need somewhere to store and index your splits so that they can later be searched. This is often done using a VectorStore and Embeddings model.

![Indexing process](./assets/images/WEE3pjeJvSZP0R7UL7CYTA.png)

* **Retrieval and generation**
**Retrieve:** Given a user input, relevant splits are retrieved from storage using a retriever.
**Generate:** A ChatModel / LLM produces an answer using a prompt that includes the question and the retrieved data.

![Indexing process](./assets/images/SwPO26VeaC8VTZwtmWh5TQ.png)




In [1]:
# 1. Gestión de Advertencias
import warnings
def warn(*args, **kwargs): pass
warnings.warn = warn
warnings.filterwarnings('ignore')

# 2. Carga y Procesamiento de Documentos (Community & Splitters)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

# 3. Vector Store y Embeddings (Paquetes específicos)
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 4. Motor de IA: Google Gemini (Reemplaza a WatsonX)
from langchain_google_genai import ChatGoogleGenerativeAI


# 5. Cadenas, Prompts y Memoria (Core de LangChain)
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

'''
from langchain.chains.retrieval_qa.base import RetrievalQA
from langchain.chains import ConversationalRetrievalChain
from langchain_core.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
'''

# 6. Utilidades
import wget
import os

!pip list | grep langchain

# Carga automática de tu .env de infraestructura
#load_dotenv(dotenv_path='../environments/.env')
print(os.getenv("Gemini_PROJECT_ID"))

langchain                                1.2.8
langchain-chroma                         1.1.0
langchain-classic                        1.0.1
langchain-community                      0.4.1
langchain-core                           1.2.8
langchain-google-genai                   4.2.0
langchain-huggingface                    1.2.0
langchain-text-splitters                 1.1.0
AI_KNOWLEDGE_T1


# Load the document

In [2]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Use wget to download the file
wget.download(url, out=filename)
print('file downloaded')

file downloaded


In [3]:
# Print the doc
with open(filename, 'r') as file:
    # Read the contents of the file
    contents = file.read()
    print(contents)
    

1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.
Accountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our practices. We report any potential violations of 

# Splitting the document into chunks

In [4]:
loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
print(len(texts))

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16


# Embedding and storing
In this step, you're taking the pieces of the story, your "chunks," converting the text into numbers, and making them easier for your computer to understand and remember by using a process called "embedding." Think of embedding like giving each chunk its own special code. This code helps the computer quickly find and recognize each chunk later on.

The following code creates a default embedding model from Hugging Face and ingests them to Chromadb.


In [5]:
embeddings = HuggingFaceEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings)  # store the embedding in docsearch using Chromadb
print('document ingested')

document ingested


# LLM model construction

In [8]:
# llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", google_api_key=os.getenv("GEMINI_APIKEY"))
llm = ChatGoogleGenerativeAI(
    model="gemini-3-pro-preview",
    api_key = os.getenv("GEMINI_APIKEY"),
    temperature=1.0,  # Gemini 3.0+ defaults to 1.0
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

In [18]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
print(ai_msg.text)

J'adore la programmation.


# Integrating LangChain
LangChain has a number of components that are designed to help retrieve information from the document and build question-answering applications, which helps you complete the retrieve part of the Retrieval task.

In [33]:
system_prompt = (
    "Use the following pieces of retrieved context to answer the question."
    " If you don't know the answer, say that you don't know."
    "\n\n"
    "{context}"
)
chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

document_chain = create_stuff_documents_chain(llm,chat_prompt)

retrieval_chain = create_retrieval_chain(
    docsearch.as_retriever(), 
    llm
)

In [37]:
response = retrieval_chain.invoke("Podrias resumir el archivo por mi?")
print(response.text)

ValueError: The input to RunnablePassthrough.assign() must be a dict.